# Human Evaluation — Statistical Analysis

Analyzes ranking data collected via the Streamlit form.

**Conditions:**
- `H` — Human (original) response
- `B` — Base LLM (Llama-3.2-1B, not fine-tuned)
- `F` — Fine-tuned LLM (LoRA adapter)

**Tests:**
1. Friedman test — non-parametric repeated-measures test for ordinal rankings
2. Wilcoxon signed-rank pairwise post-hoc (if Friedman p < 0.05), Bonferroni corrected
3. Effect size: Kendall's W
4. Per-story breakdown

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import wilcoxon
import matplotlib.pyplot as plt
from itertools import combinations

## 1. Load Data

Option A: read directly from Google Sheets via gspread.  
Option B: export the sheet as CSV and load locally.

In [ ]:
# ── OPTION A: load from Google Sheets ───────────────────────────────────────
# Uncomment and fill in your sheet URL or name.

# import gspread
# from google.oauth2.service_account import Credentials
# import json
#
# with open("../secrets.json") as f:  # export your service account JSON here
#     sa_info = json.load(f)
# creds = Credentials.from_service_account_info(
#     sa_info,
#     scopes=["https://www.googleapis.com/auth/spreadsheets", "https://www.googleapis.com/auth/drive"]
# )
# client = gspread.authorize(creds)
# ws = client.open("EmpathyEvalResults").worksheet("responses")
# df_raw = pd.DataFrame(ws.get_all_records())

# ── OPTION B: load from CSV export ──────────────────────────────────────────
df_raw = pd.read_csv("responses.csv")  # export from Google Sheets

print(f"Rows: {len(df_raw)}  |  Expected: {len(df_raw['session_id'].unique())} evaluators × 5 stories")
df_raw.head()

## 2. Reshape to Long Form

In [ ]:
rows = []
for _, r in df_raw.iterrows():
    for label in ["A", "B", "C"]:
        rows.append({
            "session_id": r["session_id"],
            "story_id":   r["story_id"],
            "condition":  r[f"cond_{label}"],   # H / B / F
            "rank":       int(r[f"rank_{label}"]),  # 1 / 2 / 3
        })

df = pd.DataFrame(rows)
print(f"Long-form rows: {len(df)}")
df.head(9)

## 3. Descriptive Statistics

In [ ]:
n_evaluators = df["session_id"].nunique()
print(f"Evaluators: {n_evaluators}")

mean_ranks = df.groupby("condition")["rank"].mean().reindex(["H", "B", "F"])
std_ranks  = df.groupby("condition")["rank"].std().reindex(["H", "B", "F"])

desc = pd.DataFrame({"mean_rank": mean_ranks, "std": std_ranks})
desc.index.name = "condition"
print("\nMean rank per condition (lower = better):")
print(desc.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
colors = {"H": "#4e79a7", "B": "#f28e2b", "F": "#59a14f"}
labels = {"H": "Human", "B": "Base LLM", "F": "Fine-tuned LLM"}

bars = ax.barh(
    [labels[c] for c in ["H", "B", "F"]],
    [mean_ranks[c] for c in ["H", "B", "F"]],
    xerr=[std_ranks[c] for c in ["H", "B", "F"]],
    color=[colors[c] for c in ["H", "B", "F"]],
    capsize=4,
    height=0.5,
)
ax.set_xlabel("Mean Rank (lower = better)")
ax.set_title("Human Evaluation: Mean Rank per Condition")
ax.set_xlim(1, 3)
ax.axvline(2, color="gray", linestyle="--", linewidth=0.8, alpha=0.5)
plt.tight_layout()
plt.savefig("mean_ranks.png", dpi=150)
plt.show()

## 4. Friedman Test

Non-parametric repeated-measures test for ordinal data.  
**H0**: All three conditions have the same rank distribution.  
Aggregate to mean rank per (evaluator, condition) across all 5 stories.

In [ ]:
# Mean rank per evaluator per condition (aggregated across 5 stories)
pivot = (
    df.groupby(["session_id", "condition"])["rank"]
    .mean()
    .unstack("condition")
    .reindex(columns=["H", "B", "F"])
    .dropna()
)
print(f"Pivot shape: {pivot.shape}  (evaluators × conditions)")

stat, p_value = stats.friedmanchisquare(pivot["H"], pivot["B"], pivot["F"])

# Kendall's W (effect size)
k = 3  # conditions
N = len(pivot)
kendall_w = stat / (N * (k - 1))

print(f"\nFriedman chi² = {stat:.4f}")
print(f"p-value       = {p_value:.4f}")
print(f"Kendall's W   = {kendall_w:.4f}  (effect size; 0=none, 1=perfect agreement)")
print()
if p_value < 0.05:
    print("✓ Significant difference between conditions (p < 0.05). Running post-hoc tests.")
else:
    print("✗ No significant difference detected (p ≥ 0.05).")

## 5. Wilcoxon Signed-Rank Post-hoc (Bonferroni corrected)

Paired test — same evaluators rated all conditions.  
3 comparisons → Bonferroni threshold = 0.05 / 3 = 0.0167.

In [ ]:
alpha = 0.05
pairs = list(combinations(["H", "B", "F"], 2))
bonferroni = alpha / len(pairs)

results = []
for c1, c2 in pairs:
    try:
        stat_w, p_w = wilcoxon(pivot[c1].values, pivot[c2].values)
    except ValueError as e:
        # Wilcoxon fails if all differences are zero
        stat_w, p_w = float("nan"), float("nan")
    results.append({
        "comparison": f"{c1} vs {c2}",
        "W_statistic": stat_w,
        "p_value": p_w,
        "significant (Bonferroni)": p_w < bonferroni if not pd.isna(p_w) else False,
    })

posthoc_df = pd.DataFrame(results)
print(f"Bonferroni threshold: {bonferroni:.4f}\n")
print(posthoc_df.to_string(index=False))

## 6. Per-Story Breakdown

Sanity check: are results consistent across stories, or driven by one outlier?

In [ ]:
per_story = (
    df.groupby(["story_id", "condition"])["rank"]
    .mean()
    .unstack("condition")
    .reindex(columns=["H", "B", "F"])
)
print("Mean rank per story per condition (lower = better):")
print(per_story.to_string())

In [ ]:
fig, axes = plt.subplots(1, len(per_story), figsize=(3 * len(per_story), 4), sharey=True)
for ax, (story_id, row) in zip(axes, per_story.iterrows()):
    ax.bar(
        [labels[c] for c in ["H", "B", "F"]],
        [row[c] for c in ["H", "B", "F"]],
        color=[colors[c] for c in ["H", "B", "F"]],
    )
    ax.set_title(f"Story {story_id}")
    ax.set_ylim(1, 3)
    ax.set_ylabel("Mean Rank" if ax == axes[0] else "")
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("Mean Rank per Story per Condition", y=1.02)
plt.tight_layout()
plt.savefig("per_story_ranks.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Summary

In [ ]:
print("=" * 50)
print("HUMAN EVALUATION SUMMARY")
print("=" * 50)
print(f"Evaluators:       {N}")
print(f"Stories:          5")
print(f"Conditions:       H=Human, B=Base LLM, F=Fine-tuned LLM")
print()
print("Mean ranks (lower = better preferred by evaluators):")
for c in ["H", "B", "F"]:
    print(f"  {labels[c]:20s}  {mean_ranks[c]:.3f} ± {std_ranks[c]:.3f}")
print()
print(f"Friedman chi² = {stat:.4f},  p = {p_value:.4f},  Kendall's W = {kendall_w:.4f}")
print()
print("Pairwise Wilcoxon (Bonferroni α = 0.0167):")
for _, row in posthoc_df.iterrows():
    sig = "*" if row["significant (Bonferroni)"] else " "
    print(f"  {sig} {row['comparison']:12s}  W={row['W_statistic']:.1f}  p={row['p_value']:.4f}")
print("=" * 50)